# Next Observed — prediction associations

Find **correlations and associations in NOI forecasts** among indicators and partners.

Not clustering. Indicator grouping lives in `IndicatorClustering.ipynb`.

1. **Same IOC, other partner** — if predicted/active at A, what is the 7-day probability at B?
2. **IOC co-occurrence** — which indicators are forecast together at the same partner?
3. **Partner correlation** — which OpDivs have similar prediction profiles?

Latest production day under `OpDiv_Predictions`. Outputs: `htoc_ml/analysis/_outputs/next_observed_relationships/`


## Load today's NOI forecasts


In [ ]:
import pandas as pd
from pathlib import Path
from datetime import date
from concurrent.futures import ThreadPoolExecutor, as_completed
from htoc.core.eval.metrics import parse_probability_percent

def load_one(path: Path) -> pd.DataFrame:
    return pd.read_csv(path).assign(Partner=path.parent.name)

main_folder = Path(r"Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions")
allowed_folders = ["CDC", "CMS", "DHA", "FDA", "HHS", "HRSA", "IHS", "NIH", "OS", "VA"]
file_date = date.today().strftime("%Y%m%d")
paths = [
    main_folder / p / f"{p}_output_{file_date}.csv"
    for p in allowed_folders
    if (main_folder / p / f"{p}_output_{file_date}.csv").is_file()
]
if not paths:
    raise FileNotFoundError(f"No partner CSVs for {file_date} under {main_folder}")

frames = []
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(load_one, path): path for path in paths}
    for fut in as_completed(futures):
        frames.append(fut.result())

NOI_df = pd.concat(frames, ignore_index=True)
NOI_df = NOI_df.rename(columns={"ensemble_45d": "Probability: 45-Day"})
PROB_SRC = {
    "noi_prob_1": "Probability: 1-Day",
    "noi_prob_7": "Probability: 7-Day",
    "noi_prob_14": "Probability: 14-Day",
    "noi_prob_30": "Probability: 30-Day",
    "noi_prob_45": "Probability: 45-Day",
}
for col, src in PROB_SRC.items():
    if src in NOI_df.columns:
        NOI_df[col] = parse_probability_percent(NOI_df[src]) / 100.0

NOI_df["Indicator"] = NOI_df["Indicator"].astype("string").str.strip()
NOI_df["Partner"] = NOI_df["Partner"].astype("string").str.strip()
print(f"{len(paths)} partner files for {file_date}: {NOI_df.shape}")
NOI_df.head()


## Indicator × partner probability matrix


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.close("all")

noi = NOI_df.copy()
noi["active_7d"] = noi["Observed Today"].eq(1) | (noi["Frequency (7d)"].fillna(0) > 0)

prob_mat = (
    noi.pivot_table(index="Indicator", columns="Partner", values="noi_prob_7", aggfunc="max")
    .sort_index(axis=1)
)
active_mat = (
    noi.pivot_table(index="Indicator", columns="Partner", values="active_7d", aggfunc="max")
    .fillna(False)
    .astype(bool)
    .reindex(columns=prob_mat.columns)
)
print(f"probability matrix {prob_mat.shape[0]} indicators × {prob_mat.shape[1]} partners")
display(prob_mat.describe().round(3))


## Partner–partner correlation

Each partner is a vector of 7-day probabilities over the same indicators.
High correlation means those OpDivs get similar NOI scores for the same IOCs.


In [ ]:
partner_corr = prob_mat.corr(min_periods=30)
display(partner_corr.round(3))

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(partner_corr.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
labs = list(partner_corr.columns)
ax.set_xticks(range(len(labs)))
ax.set_yticks(range(len(labs)))
ax.set_xticklabels(labs, rotation=45, ha="right")
ax.set_yticklabels(labs)
ax.set_title("Partner correlation of 7-day NOI probabilities")
fig.colorbar(im, ax=ax, label="Pearson r")
fig.tight_layout()
plt.show()
plt.close(fig)


## Same indicator: active at source → probability at target

Next-observed pattern: IOC is **active** at source (seen today or freq_7 > 0) and we look at
the forecast probability at every other partner. High mean `target_noi_prob_7` is the
association "seen at A, likely next at B."


In [ ]:
HORIZON_DAYS = 7
rows = []
partners = list(prob_mat.columns)
for indicator, row in active_mat.iterrows():
    sources = [p for p in partners if bool(row.get(p, False))]
    if not sources:
        continue
    probs = prob_mat.loc[indicator]
    for source in sources:
        for target in partners:
            if target == source:
                continue
            if bool(row.get(target, False)):
                continue
            val = probs.get(target)
            if pd.isna(val):
                continue
            rows.append(
                {
                    "indicator": indicator,
                    "source": source,
                    "target": target,
                    "target_noi_prob_7": float(val),
                    "horizon_days": HORIZON_DAYS,
                }
            )

spread = pd.DataFrame(rows)
print(f"{len(spread):,} directed pairs (active at source, not at target)")

pair_assoc = (
    spread.groupby(["source", "target"], as_index=False)
    .agg(n=("indicator", "size"), mean_target_noi=("target_noi_prob_7", "mean"))
    .sort_values(["mean_target_noi", "n"], ascending=False)
)
display(pair_assoc.head(20).round(3))

heat = pair_assoc.pivot(index="source", columns="target", values="mean_target_noi")
heat = heat.reindex(index=partners, columns=partners)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(heat.to_numpy(float), cmap="YlOrRd", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(partners)))
ax.set_yticks(range(len(partners)))
ax.set_xticklabels(partners, rotation=45, ha="right")
ax.set_yticklabels(partners)
ax.set_xlabel("Target")
ax.set_ylabel("Source")
ax.set_title("Mean 7-day NOI at target | active at source, absent at target")
fig.colorbar(im, ax=ax, label="mean noi_prob_7")
fig.tight_layout()
plt.show()
plt.close(fig)


## Indicator co-occurrence at the same partner

Among IOCs **active** at a partner, which pairs appear together?
Lift > 1 means they co-occur more often than independently.
Restricted to indicators active at 2+ partners so rare one-offs do not dominate.


In [ ]:
from itertools import combinations

active_inds = active_mat.index[active_mat.sum(axis=1) >= 2]
am = active_mat.loc[active_inds]
n_partners = am.shape[1]
support = am.sum(axis=1) / n_partners

pair_rows = []
for partner in am.columns:
    present = am.index[am[partner]].tolist()
    if len(present) < 2:
        continue
    for a, b in combinations(present, 2):
        pair_rows.append((a, b) if a < b else (b, a))

co = pd.Series(pair_rows).value_counts()
assoc = pd.DataFrame(
    [(a, b, n) for (a, b), n in co.items()],
    columns=["indicator_a", "indicator_b", "n_partners_together"],
)
assoc["support_a"] = assoc["indicator_a"].map(support)
assoc["support_b"] = assoc["indicator_b"].map(support)
assoc["lift"] = assoc["n_partners_together"] / n_partners / (
    assoc["support_a"] * assoc["support_b"]
)
assoc = assoc.sort_values(["lift", "n_partners_together"], ascending=False)
print(f"{len(assoc):,} co-occurring indicator pairs")
display(assoc.head(20).round(3))


## Indicator–indicator correlation (shared partner profile)

Pearson correlation of 7-day probability vectors across partners, for IOCs present
at 3+ partners. High r means two indicators get similar NOI scores at the same OpDivs.


In [ ]:
wide = prob_mat.loc[active_mat.sum(axis=1) >= 3]
# cap size so the corr matrix stays readable
top = (
    wide.mean(axis=1)
    .sort_values(ascending=False)
    .head(40)
    .index
)
ind_corr = wide.loc[top].T.corr(min_periods=3)
display(ind_corr.round(2).iloc[:10, :10])

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(ind_corr.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_title("Top-40 indicators: correlation of 7-day probs across partners")
ax.set_xticks([])
ax.set_yticks([])
fig.colorbar(im, ax=ax, label="Pearson r")
fig.tight_layout()
plt.show()
plt.close(fig)

# strongest associated pairs
corr_pairs = (
    ind_corr.where(np.triu(np.ones(ind_corr.shape), k=1).astype(bool))
    .stack()
    .rename("r")
    .reset_index()
)
corr_pairs.columns = ["indicator_a", "indicator_b", "r"]
display(corr_pairs.sort_values("r", ascending=False).head(15).round(3))


## Save


In [ ]:
out = Path(r"h:\HTOC\htoc_ml\analysis\_outputs\next_observed_relationships")
out.mkdir(parents=True, exist_ok=True)
partner_corr.to_csv(out / "partner_prob_correlation.csv")
pair_assoc.to_csv(out / "source_target_mean_noi.csv", index=False)
spread.to_csv(out / "spread_candidates_today.csv", index=False)
assoc.head(500).to_csv(out / "indicator_cooccurrence_top.csv", index=False)
corr_pairs.to_csv(out / "indicator_prob_correlation_pairs.csv", index=False)
print("wrote", out)
